# 02 — Calibration

**Project:** *Selling Property Rental Ownership — A Hybrid Real Estate Model*  
**Author:** Dan Allouche · **Date:** May 2026

This notebook calibrates the three market-dynamics models used downstream:

1. **GBM** for the Paris residential price level, fitted to the Notaires-INSEE
   quarterly index via closed-form MLE on log-returns.
2. **Merton (1976) jump-diffusion** on the same series, optimised numerically
   on the Poisson-mixture log-density.
3. **Vasicek (1977) short rate** on the OAT 10Y monthly series, fitted by
   exact MLE on the Gaussian transition.

Outputs:

- A formatted summary table of fitted parameters with standard errors and AIC/BIC.
- The four-panel diagnostic figure (`fig_calibration_diagnostics`) saved under `artifacts/figures/`.
- `data/processed/calibrated_params.yaml`, the canonical input of the Monte Carlo engine downstream.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from tranche_pricing.data import insee_irl, notaires, oat
from tranche_pricing.calibration import mle_gbm, mle_jump, mle_vasicek, runner
from tranche_pricing.viz import figures
from tranche_pricing.viz.style import apply_style

apply_style()
pd.options.display.float_format = "{:,.4f}".format


## Load the underlying data

In [ ]:
paris = notaires.fetch()
oat_df = oat.fetch()
irl_df = insee_irl.fetch()

summary = pd.DataFrame(
    {
        "series": [notaires.LABEL, oat.LABEL, insee_irl.LABEL],
        "n_obs": [len(paris), len(oat_df), len(irl_df)],
        "start": [paris["date"].min().date(), oat_df["date"].min().date(), irl_df["date"].min().date()],
        "end": [paris["date"].max().date(), oat_df["date"].max().date(), irl_df["date"].max().date()],
    }
)
display(summary)


## GBM and Merton on the Paris price index

Both models are estimated on the quarterly log-return series. AIC / BIC select the more parsimonious one.

In [ ]:
r_paris = np.diff(np.log(paris["price_index"].dropna().values))
fit_gbm = mle_gbm.calibrate(r_paris, dt=0.25)
fit_merton = mle_jump.calibrate(r_paris, dt=0.25)

gbm_p = fit_gbm.params
m_p = fit_merton.params
display(
    pd.DataFrame(
        [
            {
                "model": "GBM",
                "mu": gbm_p.mu,
                "sigma": gbm_p.sigma,
                "lambda": "—",
                "mu_J": "—",
                "sigma_J": "—",
                "log_lik": fit_gbm.log_likelihood,
                "AIC": fit_gbm.aic,
                "BIC": fit_gbm.bic,
            },
            {
                "model": "Merton",
                "mu": m_p.mu,
                "sigma": m_p.sigma,
                "lambda": m_p.lam,
                "mu_J": m_p.mu_jump,
                "sigma_J": m_p.sigma_jump,
                "log_lik": fit_merton.log_likelihood,
                "AIC": fit_merton.aic,
                "BIC": fit_merton.bic,
            },
        ]
    )
)


## Vasicek on the OAT 10Y short rate

In [ ]:
oat_dec = oat_df["yield_pct"].dropna().values / 100.0
fit_v = mle_vasicek.calibrate(oat_dec, dt=1.0 / 12.0)
vas_p = fit_v.params
display(
    pd.DataFrame(
        [
            {
                "model": "Vasicek",
                "a": vas_p.a,
                "b": vas_p.b,
                "sigma_r": vas_p.sigma_r,
                "half_life_y": fit_v.extra["half_life_years"],
                "log_lik": fit_v.log_likelihood,
                "AIC": fit_v.aic,
            }
        ]
    )
)


## Figure #13 — Calibration diagnostics

Four panels: (a) Paris log-return density vs the fitted Gaussian, (b) the corresponding QQ-plot, (c) Vasicek AR(1) residuals, (d) the ACF of squared Paris log-returns up to lag 20 (significant first-lags indicate volatility clustering, which is exactly the empirical signature the Merton extension addresses).

In [ ]:
alpha = fit_v.extra["alpha"]
beta = fit_v.extra["beta"]
oat_residuals = oat_dec[1:] - (alpha + beta * oat_dec[:-1])

fig = figures.fig_calibration_diagnostics(
    paris_log_returns=pd.Series(r_paris),
    gbm_sigma=fit_gbm.params.sigma,
    gbm_dt=0.25,
    oat_residuals=pd.Series(oat_residuals),
    oat_sigma_eps=fit_v.extra["sigma_eps"],
)
fig.savefig(ROOT / "artifacts/figures/fig_calibration_diagnostics.pdf")
fig.savefig(ROOT / "artifacts/figures/fig_calibration_diagnostics.png", dpi=200)
print("Saved fig_calibration_diagnostics.{pdf,png}")


## Persist `calibrated_params.yaml`

The Monte Carlo engine downstream reads this file rather than recomputing the fits. We use the project-level runner so the YAML structure stays in sync with the CLI.

In [ ]:
payload = runner.run_all()
out = runner.persist(payload)
print(f"Wrote {out.relative_to(ROOT)}")


---
**Next step:** `03_models_ABC.ipynb` simulates the three ownership structures (A / B / C) under the GBM-Vasicek baseline calibrated here.